# Case study 2: PNW heat dome with `its2s`

ITS counterfactual on Washington weekly injury deaths (`case_study2_data_clean.ipynb`), using **Prophet + XGBoost** (`prophet_xgb`). No covariates.

- **Data:** `../data/pnw_heatwave_injury_deaths_weekly.csv` (weekly, 2014–2021)
- **Intervention:** 2021-06-25
- **Holdout:** 3 weeks post-intervention (through week starting **2021-07-10**)
- **Split:** custom `days` windows (`split_method="days"`) because the default percent split is intended for daily series


In [ ]:
# load packages 
from pathlib import Path
import logging
import warnings
import pandas as pd
from IPython.display import Image, display
from its2s import run_single_its

# ignore warnings to clean up output
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
for _name in ("cmdstanpy", "prophet", "stan", "its2s", "xgboost", "matplotlib"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# setup
DATA_PATH = Path("../data/pnw_heatwave_injury_deaths_weekly.csv")
OUT_DIR = Path("../outputs")
INTERVENTION = "2021-06-25"
HOLDOUT_LAST_WEEK = pd.Timestamp("2021-07-10")
PLOT_COLORS = ["#984136", "#c26a7a", "#ecc0a1", "#f0f0e4"]
PLOT_FONT_SIZES = {"title": 22, "axis_label": 20, "tick": 18, "legend": 18}

df = pd.read_csv(DATA_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

_intv = pd.Timestamp(INTERVENTION)
_holdout_days = (HOLDOUT_LAST_WEEK - _intv).days
_test_days = max(28, int(0.20 * (df["ds"] < _intv).sum()))

In [ ]:
# run its2s

result = run_single_its(
    df,
    intervention_date=INTERVENTION,
    config_overrides={
        "periods": {
            "split_method": "days",
            "test_days": _test_days,
            "holdout_days": _holdout_days,
        },
        "output": {
            "plot_colors": PLOT_COLORS,
            "plot_font_sizes": PLOT_FONT_SIZES,
        },
    },
    output_dir=OUT_DIR,
)

print(result.summary())

## Excess during 3-week holdout


In [ ]:
# results 

row = result.excess_table.period_excess.iloc[0]
# period CIs are on total excess (counts); convert to % for display
pct_lo = 100 * row["excess_ci_lo"] / row["total_expected"]
pct_hi = 100 * row["excess_ci_hi"] / row["total_expected"]

display(result.excess_table.obs_excess.round(1))

print(
    f"Total excess: {row['total_excess']:.0f} "
    f"[{row['excess_ci_lo']:.0f}, {row['excess_ci_hi']:.0f}]  |  "
    f"Percent excess vs expected: {row['excess_pct']:+.1f}% "
    f"[{pct_lo:+.1f}%, {pct_hi:+.1f}%]"
)

In [ ]:
display(Image(filename=OUT_DIR / f"{result.model_name}_counterfactual.png"))